In [6]:
from lib.hw_build import HwBuildHelper
from lib.sw_build import SwBuildHelper

In [7]:
board_build_tcl = "./kv260/board_build.tcl"
constraint_xdc  = "./kv260/constraint.xdc"

In [8]:
# ── Streamer definitions ────────────────────────────────────────────────────
# Index 0 is always the DMA pass-through streamer.
# Each entry: load_width / store_width in bytes (must be power of two),
#             actual_width in bits (<= load/store_width * 8),
#             amount_row: BRAM depth.
dfx_streamers = [
    {"load_width": 4, "store_width": 4, "actual_width": 32, "amount_row": 1024},  # streamer 0 (DMA)
    {"load_width": 4, "store_width": 4, "actual_width": 32, "amount_row": 1024},  # streamer 1 (MGS 1)
    {"load_width": 4, "store_width": 4, "actual_width": 32, "amount_row": 1024},  # streamer 2 (MGS 2)
]

# ── Region definitions ──────────────────────────────────────────────────────
# Two reconfigurable regions forming a pipeline: DMA → region 0 (RM0) → region 1 (RM0) → region 0 (RM1) → region 1 (RM1) → DMA
# load_streamers / store_streamers: list of streamer indices connected to this region.
dfx_regions = [
    {"load_streamers": [0, 2], "store_streamers": [1]   },  # region 0: rm0 loads s0, stores s1  ||| rm1 load s2, stores s1
    {"load_streamers": [1, 2]   , "store_streamers": [0, 2]},     # region 1: rm0 loads s1, stores s2  ||| rm1 load s2, stores s0
]

# ── Reconfigurable module (RM) schematics ───────────────────────────────────
# 2-D list: rm_schemetics[region_idx][rm_idx]
# load_io_map / store_io_map: list of (streamer_index, kernel_port_index) pairs.
# io_idx must be in the region's declared load_streamers / store_streamers.
rm_schemetics = [
    [  # region 0: loads from s0, stores to s1
        {"load_io_map": [(0, 0)], "store_io_map": [(1, 0)]},  # rm_0
        {"load_io_map": [(2, 0)], "store_io_map": [(1, 0)]},  # rm_1
    ],
    [  # region 1: loads from s1, stores to s0
        {"load_io_map": [(1, 0)], "store_io_map": [(2, 0)]},  # rm_0
        {"load_io_map": [(1, 0)], "store_io_map": [(0, 0)]},  # rm_0
    ]
]

# ── Instantiate HwBuildHelper ───────────────────────────────────────────────
hw_builder = HwBuildHelper(
    build_folder_path="./build_prj",
    dfx_root_path="../..",
    board="custom",
    board_build_tcl=board_build_tcl,
    constraint_xdc=constraint_xdc,
    user_repo_path="",
    user_rm_build_tcl_path="",
    req_gen_ip=1,
    num_core=4,
    clk_frq=99999001,          # Hz
    rm_index_width=3,           # 1 << rm_index_width = max bank-1 slots
    dfx_streamers=dfx_streamers,
    dfx_regions=dfx_regions,
    rm_schemetics=rm_schemetics,
    test_mode=1,
    vivado_path="/tools/Xilinx/Vivado/2023.2/bin/vivado",
    export_folder_path="./export"
)

In [9]:
hw_builder.run_build()


Running Vivado with /media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/example/multi_region_test3/build_prj/run_build.tcl...

****** Vivado v2023.2 (64-bit)
  **** SW Build 4029153 on Fri Oct 13 20:13:54 MDT 2023
  **** IP Build 4028589 on Sat Oct 14 00:45:43 MDT 2023
  **** SharedData Build 4025554 on Tue Oct 10 17:18:54 MDT 2023
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.

start_gui
INFO: [Common 17-206] Exiting Vivado at Fri May 22 23:12:21 2026...


In [10]:
hw_builder.package_export_files()

FileNotFoundError: [Errno 2] No such file or directory: '/media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/example/multi_region_test3/build_prj/link_prj/link_prj.runs/child_0_impl_dfx/dfx4ml_i_dfx_pr_region_0_0_dfx_pr_region_0_rm_0_inst_0_partial.bin'

In [ ]:
sw_builder = SwBuildHelper(export_folder_path="./export", num_pr_region = 2, rm_index_width = 3, num_streamer = 3)

In [ ]:
sw_builder.package_export_file()